# 4. Predicted against ground-truth sub-regions

Reproduces **Section 4.4** and `tab:abl`: does the grading signal depend on the ground-truth
annotations, or does it survive on the sub-regions the method estimates for itself?

The two arms differ in exactly one thing. Both compute the same seventeen features with the same
function under the same protocol; one reads sub-regions estimated from the enhancement map, the
other reads them off the dataset's label codes (`seg == 4`, `seg == 1`, `seg == 2`, `seg > 0`).
Nothing is estimated on the ground-truth side.

Both arms are evaluated on the **same matched case list**. Scoring the predicted arm on its own
full 369 instead would break the pairing and make the differences meaningless.

Runs from `data_cache/` alone.

Sign convention, as in the paper: $\Delta$ = predicted minus ground truth.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, tables

predicted = cohort.extract_features()
ground_truth = cohort.extract_gt_features()
grades = cohort.load_grade_labels()

missing = sorted(set(predicted) - set(ground_truth))
print(f"predicted {len(predicted)} cases | ground truth {len(ground_truth)} cases")
print(f"no reference mask, so out of this comparison: {missing}")

## Wiring proof on the superseded cohort

The older version of this table was on 367 cases: the one case that genuinely has no reference
mask, plus the one an earlier run lost to an unreadable file. Reproducing it shows the protocol is
unchanged. The largest deviation sits on the random forest and comes from library versions.

In [ ]:
superseded = {"SVM (RBF)": (0.950, 0.925), "SVM (linear)": (0.904, 0.920),
              "Random forest": (0.953, 0.936), "Gradient boosting": (0.933, 0.933),
              "Logistic reg.": (0.890, 0.924), "$k$-NN ($k = 5$)": (0.907, 0.886)}

table_367, matched_367, _ = tables.table_abl(
    ground_truth, predicted, grades,
    exclude={cohort.NO_GROUND_TRUTH, cohort.PREVIOUSLY_SKIPPED})
print(f"n = {len(matched_367)}")
display(table_367)
display(tables.compare_to_paper(
    table_367, "Ground truth", {k: v[0] for k, v in superseded.items()}, tolerance=0.005))

## tab:abl

Expected, on the 368-case cohort:

| Classifier | Ground truth | Predicted | $\Delta$ |
|---|---|---|---|
| SVM (RBF) | 0.947 | 0.928 | -0.019 |
| SVM (linear) | 0.907 | 0.925 | +0.019 |
| Random forest | 0.951 | 0.931 | -0.021 |
| Gradient boosting | 0.933 | 0.914 | -0.019 |
| Logistic reg. | 0.894 | 0.921 | +0.028 |
| k-NN (k = 5) | 0.914 | 0.894 | -0.020 |

The AUC shifts by at most 0.028 in either direction and two of the six classifiers score higher
on the estimated sub-regions.

In [ ]:
table_368, matched_368, y_368 = tables.table_abl(ground_truth, predicted, grades)
assert len(matched_368) == 368
assert cohort.PREVIOUSLY_SKIPPED in matched_368
assert cohort.NO_GROUND_TRUTH not in matched_368
print(f"n = {len(matched_368)}  high-grade {int((y_368 == 0).sum())}  "
      f"low-grade {int(y_368.sum())}")
display(table_368)

In [ ]:
delta = table_368["$\\Delta$"]
checks = pd.Series({
    "largest shift in either direction": delta.abs().max(),
    "classifiers scoring higher on the estimated sub-regions": int((delta > 0).sum()),
    "classifiers in the panel": len(delta),
    "smallest shift by magnitude": delta.abs().min(),
}, name="value")
display(checks.to_frame())

## Why the substitution changes so little

Two reasons, and the paper gives both.

**Where the segmentation errs.** The predicted mask keeps 94.8% of the enhancing tumor and 97.1%
of the necrotic core but only 88.9% of the edema (notebook 1), while the SHAP attributions
(notebook 3) put the grading signal on the enhancing and necrotic measures. What the segmentation
loses is the compartment the grade is least read from. Even in the weakest case in the cohort, at
a Dice of 0.74, 99% of the enhancing tumor and necrotic core is kept.

**The two definitions differ in kind, not in quality.** The same two thresholds are applied to
every case, whereas the reference sub-regions are drawn one case at a time. A rule that is uniform
across the cohort can produce features a classifier separates slightly better even where it is
locally less accurate, which is a plausible reading of the two classifiers that improve. With
76 low-grade cases, shifts of 0.019 to 0.028 are the size of the cross-validation noise, so none
of them should be read as a real difference in either direction.

What the table supports is the weaker and sufficient claim: the estimated sub-regions carry the
grading signal as well as the annotated ones do.